# 위하고 엑셀 파일 테스트 

## 소득

### 근로소득

In [ ]:
from pathlib import Path
import pandas as pd

# 엑셀 파일 경로 (노트북 기준 현재 작업 폴더에서 탐색)
file_name = "종로구육아종합지원센터-202512.xlsx"
file_path = Path(file_name)

if not file_path.exists():
    # 워크스페이스 전체에서 동일 파일명 검색
    candidates = list(Path('.').rglob(file_name))
    if candidates:
        file_path = candidates[0]
        print(f"파일을 찾았습니다: {file_path}")
    else:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_name}")


def _clean(v):
    if pd.isna(v):
        return ""
    text = str(v).strip()
    if text.lower().startswith("unnamed"):
        return ""
    return text


def build_columns_from_two_header_rows(sheet_df):
    h1 = sheet_df.iloc[0].tolist()  # 1행 헤더
    h2 = sheet_df.iloc[1].tolist()  # 2행 헤더

    columns = []
    for a, b in zip(h1, h2):
        a = _clean(a)
        b = _clean(b)

        if a and b and a != b:
            col = f"{a}_{b}"   # 예: 수당_기본급
        elif b:
            col = b              # 2행 우선
        elif a:
            col = a              # 2행 비면 1행 사용
        else:
            col = ""

        columns.append(col)

    # 빈 컬럼명/중복 컬럼명 정리
    seen = {}
    normalized = []
    for idx, col in enumerate(columns, start=1):
        if not col:
            col = f"col_{idx}"

        seen[col] = seen.get(col, 0) + 1
        if seen[col] > 1:
            col = f"{col}_{seen[col]}"

        normalized.append(col)

    return normalized


# 1) 시트 목록 확인
excel_file = pd.ExcelFile(file_path)
print("시트 목록:", excel_file.sheet_names)

# 2) 첫 번째 시트: 헤더 2행 병합 후 데이터 읽기
first_sheet = excel_file.sheet_names[0]
raw_first = pd.read_excel(file_path, sheet_name=first_sheet, header=None)
columns_first = build_columns_from_two_header_rows(raw_first)

df = raw_first.iloc[2:].copy()     # 데이터는 3행부터
df.columns = columns_first
df = df.reset_index(drop=True)

print(f"\n[첫 시트: {first_sheet}] 행/열 크기:", df.shape)
print("컬럼:", df.columns.tolist())
display(df.head())

# 3) 모든 시트 상위 3행 미리보기 (헤더 2행 병합 적용)
for sheet in excel_file.sheet_names:
    raw = pd.read_excel(file_path, sheet_name=sheet, header=None)
    cols = build_columns_from_two_header_rows(raw)
    preview = raw.iloc[2:].copy()
    preview.columns = cols
    preview = preview.reset_index(drop=True).head(3)

    print(f"\n===== 시트: {sheet} =====")
    display(preview)

### 사업 소득

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

# ── 파일 경로 ──
swbu_file_name = "SWBU0102_2021_03.xls"
swbu_file_path = Path(swbu_file_name)

if not swbu_file_path.exists():
    candidates = list(Path('.').rglob(swbu_file_name))
    if candidates:
        swbu_file_path = candidates[0]
        print(f"파일을 찾았습니다: {swbu_file_path}")
    else:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {swbu_file_name}")

raw = pd.read_excel(swbu_file_path, engine="xlrd", header=None)

# ── 메타 정보(제목, 회사명) 추출 ──
title = str(raw.iloc[0, 5]).strip()       # (2025년12월) 사업소득지급대장(합계)
company = str(raw.iloc[2, 0]).strip()     # 회사명:서울유캔수학학원
print(f"제목: {title}")
print(f"{company}\n")

# ── 2행 헤더 병합 (Row 4-5) ──
h1 = raw.iloc[4].tolist()   # 1행 헤더
h2 = raw.iloc[5].tolist()   # 2행 헤더

def _clean(v):
    if pd.isna(v):
        return ""
    return str(v).strip()

# 1행만 있는 컬럼 / 2행만 있는 컬럼 / 둘 다 있는 컬럼 처리
col_row1 = []  # 1행 헤더 기준 컬럼명
col_row2 = []  # 2행 헤더 기준 컬럼명

for a, b in zip(h1, h2):
    a, b = _clean(a), _clean(b)
    col_row1.append(a if a else None)
    col_row2.append(b if b else None)

print("1행 헤더:", col_row1)
print("2행 헤더:", col_row2)

# ── 데이터 영역 (Row 6~) 을 2행씩 묶어 1인 1행으로 변환 ──
data_rows = raw.iloc[6:].reset_index(drop=True)
records = []

for i in range(0, len(data_rows), 2):
    row1 = data_rows.iloc[i]
    row2 = data_rows.iloc[i + 1] if i + 1 < len(data_rows) else pd.Series([None] * len(row1))

    record = {}
    for col_idx in range(len(col_row1)):
        name1 = col_row1[col_idx]
        name2 = col_row2[col_idx]
        val1 = row1.iloc[col_idx] if not pd.isna(row1.iloc[col_idx]) else None
        val2 = row2.iloc[col_idx] if not pd.isna(row2.iloc[col_idx]) else None

        if name1 and name2:
            # 두 행 모두 헤더가 있는 컬럼 → 각각 별도 컬럼
            record[name1] = val1
            record[name2] = val2
        elif name1:
            # 1행에만 헤더 → 1행 데이터만 사용
            record[name1] = val1
        elif name2:
            # 2행에만 헤더 → 2행 데이터만 사용
            record[name2] = val2

    records.append(record)

df_swbu = pd.DataFrame(records)

print(f"\n변환 완료: {df_swbu.shape[0]}행 × {df_swbu.shape[1]}열")
print("컬럼:", df_swbu.columns.tolist())
display(df_swbu)

파일을 찾았습니다: wehago/SWBU0102_2021_03.xls
제목: (2025년12월) 사업소득지급대장(합계)
회사명:서울유캔수학학원

1행 헤더: ['NO', '코드', '성   명', '귀속년월', '지급액', '소득세', '예술/특고인 경비', '학자금상환액', None, '영수인']
2행 헤더: [None, None, None, '지급년월', None, '지방소득세', '고용보험료', '산재보험료', '차인지급액', None]

변환 완료: 5행 × 14열
컬럼: ['NO', '코드', '성   명', '귀속년월', '지급년월', '지급액', '소득세', '지방소득세', '예술/특고인 경비', '고용보험료', '학자금상환액', '산재보험료', '차인지급액', '영수인']


,NO,코드,성 명,귀속년월,지급년월,지급액,소득세,지방소득세,예술/특고인 경비,고용보험료,학자금상환액,산재보험료,차인지급액,영수인
0,1,000006,김선례,2025.12,2025.12,7000000,210000,21000,None,None,None,None,6769000,None
1,2,000007,전영수,2025.12,2025.12,7500000,225000,22500,None,None,None,None,7252500,None
2,3,000019,김선주,2025.12,2025.12,300000,9000,900,None,None,None,None,290100,None
3,4,000020,박혜영,2025.12,2025.12,1350000,40500,4050,None,None,None,None,1305450,None
4,총 계,NaN,NaN,NaN,NaN,16150000,484500,48450,None,None,None,None,15617050,None


### 기타 소득

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# ── 파일 경로 ──
swet_file_name = "SWET0102_01_2010.xls"
swet_file_path = Path(swet_file_name)

if not swet_file_path.exists():
    candidates = list(Path('.').rglob(swet_file_name))
    if candidates:
        swet_file_path = candidates[0]
        print(f"파일을 찾았습니다: {swet_file_path}")
    else:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {swet_file_name}")

raw = pd.read_excel(swet_file_path, engine="xlrd", header=None)

# ── 메타 정보 추출 ──
title = str(raw.iloc[0, 5]).strip()       # (2025.12) 기타소득 지급 대장(합계)
period = str(raw.iloc[0, 3]).strip()      # (2025.12)
company = str(raw.iloc[2, 0]).strip()     # 종로구육아종합지원센터
print(f"제목: {title}")
print(f"기간: {period}")
print(f"회사명: {company}\n")

# ── 2행 헤더 병합 (Row 3-4) ──
h1 = raw.iloc[3].tolist()   # 1행 헤더: NO, 코드, 성명, 소득구분, 귀속년월, 지급액, 소득세/법인세, 차인지급액, 영수인
h2 = raw.iloc[4].tolist()   # 2행 헤더: 지급년월, 지방소득세

def _clean(v):
    if pd.isna(v):
        return ""
    return str(v).strip()

col_row1 = []
col_row2 = []

for a, b in zip(h1, h2):
    a, b = _clean(a), _clean(b)
    col_row1.append(a if a else None)
    col_row2.append(b if b else None)

print("1행 헤더:", col_row1)
print("2행 헤더:", col_row2)

# ── 데이터 영역 (Row 5~) 을 2행씩 묶어 1인 1행으로 변환 ──
data_rows = raw.iloc[5:].reset_index(drop=True)
records = []

for i in range(0, len(data_rows), 2):
    row1 = data_rows.iloc[i]
    row2 = data_rows.iloc[i + 1] if i + 1 < len(data_rows) else pd.Series([None] * len(row1))

    record = {}
    for col_idx in range(len(col_row1)):
        name1 = col_row1[col_idx]
        name2 = col_row2[col_idx]
        val1 = row1.iloc[col_idx] if not pd.isna(row1.iloc[col_idx]) else None
        val2 = row2.iloc[col_idx] if not pd.isna(row2.iloc[col_idx]) else None

        if name1 and name2:
            record[name1] = val1
            record[name2] = val2
        elif name1:
            record[name1] = val1
        elif name2:
            record[name2] = val2

    records.append(record)

df_swet = pd.DataFrame(records)

print(f"\n변환 완료: {df_swet.shape[0]}행 × {df_swet.shape[1]}열")
print("컬럼:", df_swet.columns.tolist())
display(df_swet)

파일을 찾았습니다: wehago/SWET0102_01_2010.xls
제목: 기타소득 지급 대장(합계)
기간: (2025.12)
회사명: 종로구육아종합지원센터

1행 헤더: ['NO', '코드', '성   명', '소득구분', '귀속년월', '지급액', '소득세/법인세', '차인지급액', '영수인']
2행 헤더: [None, None, None, None, '지급년월', None, '지방소득세', None, None]

변환 완료: 4행 × 11열
컬럼: ['NO', '코드', '성   명', '소득구분', '귀속년월', '지급년월', '지급액', '소득세/법인세', '지방소득세', '차인지급액', '영수인']


,NO,코드,성 명,소득구분,귀속년월,지급년월,지급액,소득세/법인세,지방소득세,차인지급액,영수인
0,1,000160,김성화,강연료 등,2025.12,2025.12.25,120000,0,0,120000,None
1,2,000206,한영미,강연료 등,2025.12,2025.12.25,275000,22000,2200,250800,None
2,3,000153,유도연,강연료 등,2025.12,2025.12.25,300000,24000,2400,273600,None
3,총 합 계,NaN,NaN,NaN,NaN,NaN,695000,46000,4600,644400,None


### 일용직 소득

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

# ── 파일 경로 ──
swsa_file_name = "SWSA0107_2.xls"
swsa_file_path = Path(swsa_file_name)

if not swsa_file_path.exists():
    candidates = list(Path('.').rglob(swsa_file_name))
    if candidates:
        swsa_file_path = candidates[0]
        print(f"파일을 찾았습니다: {swsa_file_path}")
    else:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {swsa_file_name}")

raw = pd.read_excel(swsa_file_path, engine="xlrd", header=None)

# ── 메타 정보 추출 ──
title = str(raw.iloc[0, 15]).strip()              # 일용직 급여 지급명세서
period_belong = str(raw.iloc[2, 10]).strip()       # 귀속: 2024년03월
period_pay = str(raw.iloc[2, 18]).strip()          # 지급: 2024년03월
company = str(raw.iloc[4, 1]).strip()              # 컴포즈커피 수원연무점
report_period = str(raw.iloc[4, 7]).strip()        # 신고: 2024년04월

print(f"제목: {title}")
print(f"귀속: {period_belong} / 지급: {period_pay}")
print(f"회사명: {company} / 신고: {report_period}\n")

# ── 2행 헤더 병합 (Row 6-7) ──
h1 = raw.iloc[6].tolist()
h2 = raw.iloc[7].tolist()

def _clean(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    return str(v).strip()

col_row1 = []
col_row2 = []

for a, b in zip(h1, h2):
    a, b = _clean(a), _clean(b)
    col_row1.append(a if a else None)
    col_row2.append(b if b else None)

print("1행 헤더:", col_row1)
print("2행 헤더:", col_row2)

# ── 데이터 영역 (Row 8~) 을 2행씩 묶어 1인 1행으로 변환 ──
data_rows = raw.iloc[8:].reset_index(drop=True)
records = []

for i in range(0, len(data_rows), 2):
    row1 = data_rows.iloc[i]
    row2 = data_rows.iloc[i + 1] if i + 1 < len(data_rows) else pd.Series([None] * len(row1))

    record = {}
    for col_idx in range(len(col_row1)):
        name1 = col_row1[col_idx]
        name2 = col_row2[col_idx]

        def _val(row, idx):
            v = row.iloc[idx]
            if v is None:
                return None
            if isinstance(v, float) and np.isnan(v):
                return None
            return v

        val1 = _val(row1, col_idx)
        val2 = _val(row2, col_idx)

        if name1 and name2:
            record[name1] = val1
            record[name2] = val2
        elif name1:
            record[name1] = val1
        elif name2:
            record[name2] = val2

    records.append(record)

df_swsa = pd.DataFrame(records)

print(f"\n변환 완료: {df_swsa.shape[0]}행 × {df_swsa.shape[1]}열")
print("컬럼:", df_swsa.columns.tolist())
display(df_swsa)

파일을 찾았습니다: wehago/SWSA0107_2.xls
제목: 일용직 급여 지급명세서
귀속: 2024년03월 / 지급: 2024년03월
회사명: 컴포즈커피 수원연무점 / 신고: 2024년04월

1행 헤더: ['코드', '직종', '주민등록번호', '전화번호', '입사일', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', None, '근무', '근무시간', None, '급여총액', '국민연금', '소득세', '요양보험', '영수인']
2행 헤더: ['성명', '직급', '주 소', None, None, '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31.0', '일수', '정상', '연장', '고용보험료', '건강보험료', '지방소득세', '차인지급액', None]

변환 완료: 5행 × 53열
컬럼: ['코드', '성명', '직종', '직급', '주민등록번호', '주 소', '전화번호', '입사일', '1', '16', '2', '17', '3', '18', '4', '19', '5', '20', '6', '21', '7', '22', '8', '23', '9', '24', '10', '25', '11', '26', '12', '27', '13', '28', '14', '29', '15', '30', '31.0', '근무', '일수', '근무시간', '정상', '연장', '급여총액', '고용보험료', '국민연금', '건강보험료', '소득세', '지방소득세', '요양보험', '차인지급액', '영수인']


,코드,성명,직종,직급,주민등록번호,주 소,전화번호,입사일,1,16,...,연장,급여총액,고용보험료,국민연금,건강보험료,소득세,지방소득세,요양보험,차인지급액,영수인
0,5,국지수,None,None,040727-*******,,None,2022.07.25,NaN,None,...,None,880000,7910,None,None,None,None,None,872090,None
1,9,김민정,None,None,000511-*******,,None,2022.08.31,NaN,None,...,None,876750,7850,None,None,None,None,None,868900,None
2,13,이다현,None,None,030222-*******,,None,2023.03.04,NaN,None,...,None,625000,5620,None,None,None,None,None,619380,None
3,14,변승민,None,None,010517-*******,,None,2023.05.01,NaN,None,...,None,758333,6810,None,None,None,None,None,751523,None
4,총인원,4명,None,None,NaN,NaN,None,NaN,합 계,None,...,None,3140083,28190,None,None,None,None,None,3111893,None


## 직원

### 근로 소득 직원 - 1

In [3]:
import pandas as pd
import numpy as np

file_path = "wehago/종로구육아종합지원센터_사원등록_20260312.xlsx"

xls = pd.ExcelFile(file_path)
print("시트 목록:", xls.sheet_names)

def _clean(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    text = str(v).strip()
    if text.lower().startswith("unnamed"):
        return ""
    return text

for sheet in xls.sheet_names:
    raw = pd.read_excel(xls, sheet_name=sheet, header=None)
    print(f"\n=== 시트: {sheet} ===")

    # 1행 헤더 추출
    header = [_clean(v) for v in raw.iloc[0].tolist()]
    print(f"  칼럼명 ({len([h for h in header if h])}개): {[h for h in header if h]}")

    # 데이터는 2행부터
    df = raw.iloc[1:].copy()
    df.columns = [h if h else f"col_{i+1}" for i, h in enumerate(header)]
    df = df.reset_index(drop=True)

    print(f"  데이터: {df.shape[0]}행 × {df.shape[1]}열")
    display(df)

시트 목록: ['Sheet1']

=== 시트: Sheet1 ===
  칼럼명 (48개): ['사원코드', '사원명', '내외국인구분', '거주지국코드', '거주지명', '국적코드', '국가명', '주민(외국인)등록번호', '나이', '성별', '입사일자', '퇴사일자', '거주구분', '우편번호', '주소', '부서', '직급', '직종', '호봉', '프로젝트', '현장', '건강보험대상금액', '건강보험료', '건강보험증번호', '국민연금대상금액', '국민연금료', '고용보험대상금액', '고용보험료', '고용보험적용여부', '산재보험적용여부', '장기요양보험적용여부', '생산직여부', '국외근로적용여부', '학자금상환공제자여부', '학자금상환공제액', '중소기업청년소득세감면여부', '전화번호', '휴대폰번호', '이메일', '급여이체은행', '계좌번호', '예금주', '배우자유무', '20세이하', '60세이상', '경로우대', '장애인', '부녀자']
  데이터: 29행 × 48열


/Users/yong/Documents/admin-system-saas/.venv_excel/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,사원코드,사원명,내외국인구분,거주지국코드,거주지명,국적코드,국가명,주민(외국인)등록번호,나이,성별,...,이메일,급여이체은행,계좌번호,예금주,배우자유무,20세이하,60세이상,경로우대,장애인,부녀자
0,1,류외희,내국인,KR,대한민국,KR,대한민국,641005-2684212,62,여,...,@,NaN,NaN,류외희,0,0,0,0,0,0
1,10,이신정,내국인,KR,대한민국,KR,대한민국,670114-2006025,59,여,...,@,NaN,NaN,이신정,0,0,1,1,0,1
2,19,엄지홍,내국인,KR,대한민국,KR,대한민국,811029-2079111,45,여,...,@,NaN,NaN,엄지홍,0,1,0,0,0,1
3,47,배은수,내국인,KR,대한민국,KR,대한민국,670120-2683521,59,여,...,@,NaN,NaN,배은수,0,0,0,0,0,0
4,56,노수연,내국인,KR,대한민국,KR,대한민국,680723-2030625,58,여,...,@,NaN,NaN,노수연,0,0,0,0,0,0
5,80,권희숙,내국인,KR,대한민국,KR,대한민국,680224-2047316,58,여,...,@,NaN,NaN,권희숙,0,0,0,0,0,0
6,99,천성현,내국인,KR,대한민국,KR,대한민국,670924-2783020,59,여,...,@,NaN,NaN,천성현,0,0,0,0,0,1
7,106,손영희,내국인,KR,대한민국,KR,대한민국,700325-2898810,56,여,...,@,NaN,NaN,손영희,0,0,0,0,0,0
8,107,이은지,내국인,KR,대한민국,KR,대한민국,920804-2478619,34,여,...,@,NaN,NaN,이은지,0,0,0,0,0,0
9,114,윤주현,내국인,KR,대한민국,KR,대한민국,940906-2026111,32,여,...,@,NaN,NaN,윤주현,0,0,0,0,0,0


### 근로 소득 직원 -2 

In [7]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ── 파일 경로 ──
file_path = "wehago/SWPM0108_01_2019.xls"
raw = pd.read_excel(file_path, engine="xlrd", header=None)

def _clean(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    text = str(v).strip()
    if text.lower().startswith("unnamed"):
        return ""
    return text

# ── 헤더: Row 5, Row 6, Row 7 ──
h_row5 = [_clean(v) for v in raw.iloc[5].tolist()]
h_row6 = [_clean(v) for v in raw.iloc[6].tolist()]
h_row7 = [_clean(v) for v in raw.iloc[7].tolist()]

# Row 5: 독립 헤더 (row_a에서 데이터 추출)
row5_indices = [i for i, c in enumerate(h_row5) if c]
row5_cols = [h_row5[i] for i in row5_indices]

# Row 6+7: forward-fill Row 6 (병합셀 대응) 후 Row 7과 결합
row6_ff = []  # forward-filled Row 6
last_r6 = ""
for v in h_row6:
    if v:
        last_r6 = v
    row6_ff.append(last_r6)

# Row 6+7 결합 칼럼 생성 (row_b에서 데이터 추출, 전화번호/휴대폰 특수 처리)
row67_indices = []  # (col_idx, col_name, source) — source: 'b' or 'c'
phone_col_idx = None  # 전화번호휴대폰이 있는 열 인덱스

for col_idx in range(len(h_row6)):
    r6 = h_row6[col_idx]
    r6f = row6_ff[col_idx]
    r7 = h_row7[col_idx]

    if r6 == "전화번호휴대폰" or (r6f == "전화번호휴대폰" and r7):
        # 전화번호/휴대폰 특수 처리: row_b → 전화번호, row_c → 휴대폰
        phone_col_idx = col_idx
        continue

    if r7:
        # Row 7에 값 있으면 → "Row6부모_Row7자식"
        col_name = f"{r6f}_{r7}" if r6f else r7
        row67_indices.append((col_idx, col_name, 'b'))
    elif r6:
        # Row 6에만 값 있고 Row 7 비어있으면 → Row 6 값 그대로
        col_name = r6
        row67_indices.append((col_idx, col_name, 'b'))

# 최종 칼럼 조합
all_cols = row5_cols + [name for _, name, _ in row67_indices]
if phone_col_idx is not None:
    all_cols += ["전화번호", "휴대폰"]

# 중복 칼럼명 정리
seen = {}
normalized = []
for col in all_cols:
    seen[col] = seen.get(col, 0) + 1
    if seen[col] > 1:
        normalized.append(f"{col}_{seen[col]}")
    else:
        normalized.append(col)

print("Row 5 칼럼:", row5_cols)
print("Row 6+7 칼럼:", [name for _, name, _ in row67_indices])
if phone_col_idx is not None:
    print(f"전화번호/휴대폰 col_idx: {phone_col_idx}")
print(f"\n최종 칼럼 ({len(normalized)}개):", normalized)

# ── 데이터: 3행씩 묶어 1행으로 ──
data_rows = raw.iloc[8:].reset_index(drop=True)
records = []

def _val(v):
    if v is None:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    return v

for i in range(0, len(data_rows), 3):
    row_a = data_rows.iloc[i]
    row_b = data_rows.iloc[i + 1] if i + 1 < len(data_rows) else pd.Series([None] * len(row_a))
    row_c = data_rows.iloc[i + 2] if i + 2 < len(data_rows) else pd.Series([None] * len(row_a))

    vals = [_val(row_a.iloc[idx]) for idx in row5_indices] + \
           [_val(row_b.iloc[idx]) for idx, _, _ in row67_indices]
    if phone_col_idx is not None:
        vals.append(_val(row_b.iloc[phone_col_idx]))  # 전화번호
        vals.append(_val(row_c.iloc[phone_col_idx]))  # 휴대폰
    records.append(vals)

df_swpm = pd.DataFrame(records, columns=normalized)

print(f"\n변환 완료: {df_swpm.shape[0]}행 × {df_swpm.shape[1]}열")
display(HTML(df_swpm.to_html()))

Row 5 칼럼: ['NO', '사원코드', '사원명', '입사일', '퇴사일', '내·외국인', '주민(외국인)등록번호', '직종구분', '거주구분', '주소', '우편번호']
Row 6+7 칼럼: ['부서', '직책명', '배우자\n유무', '부양 가족 공제_20세이하', '부양 가족 공제_60세이상', '추  가  공  제_경로우대', '추  가  공  제_장애인', '추  가  공  제_부녀자', '추  가  공  제_자녀', '추  가  공  제_출산입양', '추  가  공  제_한부모', '고용보험', '호봉', '국민연금', '건강보험']
전화번호/휴대폰 col_idx: 16

최종 칼럼 (28개): ['NO', '사원코드', '사원명', '입사일', '퇴사일', '내·외국인', '주민(외국인)등록번호', '직종구분', '거주구분', '주소', '우편번호', '부서', '직책명', '배우자\n유무', '부양 가족 공제_20세이하', '부양 가족 공제_60세이상', '추  가  공  제_경로우대', '추  가  공  제_장애인', '추  가  공  제_부녀자', '추  가  공  제_자녀', '추  가  공  제_출산입양', '추  가  공  제_한부모', '고용보험', '호봉', '국민연금', '건강보험', '전화번호', '휴대폰']

변환 완료: 29행 × 28열


,NO,사원코드,사원명,입사일,퇴사일,내·외국인,주민(외국인)등록번호,직종구분,거주구분,주소,우편번호,부서,직책명,배우자 유무,부양 가족 공제_20세이하,부양 가족 공제_60세이상,추 가 공 제_경로우대,추 가 공 제_장애인,추 가 공 제_부녀자,추 가 공 제_자녀,추 가 공 제_출산입양,추 가 공 제_한부모,고용보험,호봉,국민연금,건강보험,전화번호,휴대폰
0,1,1,류외희,2013/10/01,NaN,내국인,641005-2684212,None,None,"서울특별시 마포구 백범로 205 104동 904호(신공덕동, 마포펜트라우스)",04194,None,직급없음,무,NaN,NaN,NaN,None,NaN,None,None,None,여,None,None,None,None,--
1,2,10,이신정,2014/08/11,NaN,내국인,670114-2006025,None,None,"서울 노원구 월계로45가길 89 302동 1208호(월계동, 월계청백3단지아파트)",01875,None,직급없음,무,NaN,1,1,None,1,None,None,None,여,None,None,None,None,--
2,3,19,엄지홍,2017/02/27,NaN,내국인,811029-2079111,None,None,"경기 고양시 덕양구 화신로 311 913동 1401호(화정동, 별빛마을9단지아파트)",10502,None,직급없음,무,1,NaN,NaN,None,1,None,None,None,여,None,None,None,None,--
3,4,47,배은수,2018/02/01,NaN,내국인,670120-2683521,None,None,"서울특별시 마포구 토정로18길 11 106동 2001호(현석동, 래미안웰스트림)",04091,None,직급없음,무,NaN,NaN,NaN,None,NaN,None,None,None,여,None,None,None,None,--
4,5,56,노수연,2018/09/27,NaN,내국인,680723-2030625,None,None,"서울특별시 성북구 길음로 74 508동303호(길음동, 길음뉴타운)",02724,None,직급없음,무,NaN,NaN,NaN,None,NaN,None,None,None,여,None,None,None,None,--
5,6,80,권희숙,2019/10/12,NaN,내국인,680224-2047316,None,None,"서울특별시 도봉구 방학로 193 15동 303호(방학동, 신동아1단지아파트)",01382,None,직급없음,무,NaN,NaN,NaN,None,NaN,None,None,None,여,None,None,None,None,--
6,7,99,천성현,2021/02/19,NaN,내국인,670924-2783020,None,None,서울시 성북구 오패산로16가길 40 106동 205호,NaN,None,직급없음,무,NaN,NaN,NaN,None,1,None,None,None,여,None,None,None,None,--
7,8,106,손영희,2021/10/01,NaN,내국인,700325-2898810,None,None,,NaN,None,직급없음,무,NaN,NaN,NaN,None,NaN,None,None,None,여,None,None,None,None,--
8,9,107,이은지,2022/01/13,NaN,내국인,920804-2478619,None,None,서울특별시 광진구 구의로6길 12-14,NaN,None,직급없음,무,NaN,NaN,NaN,None,NaN,None,None,None,여,None,None,None,None,--
9,10,114,윤주현,2022/03/01,NaN,내국인,940906-2026111,None,None,,NaN,None,직급없음,무,NaN,NaN,NaN,None,NaN,None,None,None,여,None,None,None,None,--


### 사업 소득 직원

In [8]:
from pathlib import Path
import pandas as pd

# ── 파일 경로 ──
biz_emp_file_name = "사업소득자명부.xls"
biz_emp_file_path = Path("wehago") / biz_emp_file_name

if not biz_emp_file_path.exists():
    candidates = list(Path('.').rglob(biz_emp_file_name))
    if candidates:
        biz_emp_file_path = candidates[0]
        print(f"파일을 찾았습니다: {biz_emp_file_path}")
    else:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {biz_emp_file_name}")

raw = pd.read_excel(biz_emp_file_path, engine="xlrd", header=None)

# ── 메타 정보 추출 ──
title = str(raw.iloc[0, 4]).strip()       # 사 업  소 득 자  명 부
report_date = str(raw.iloc[2, 4]).strip() # 2026-03-12
company = str(raw.iloc[4, 0]).strip()     # 서울유캔수학학원
print(f"제목: {title}")
print(f"기준일: {report_date}")
print(f"회사명: {company}\n")

# ── 2행 헤더 (Row 6-7) ──
h1 = raw.iloc[6].tolist()  # NO, 코드, 성명, 소득구분, 상호, 사업장주소, 우편번호
h2 = raw.iloc[7].tolist()  # (빈), 내/외국인, 주민등록번호, 거주구분, 사업자등록번호, 소득자주소, (빈)

def _clean(v):
    if pd.isna(v):
        return ""
    return str(v).strip().replace(" ", "")

col_row1 = [_clean(v) for v in h1]
col_row2 = [_clean(v) for v in h2]

print("1행 헤더:", col_row1)
print("2행 헤더:", col_row2)

# ── 데이터 영역 (Row 8~) 2행씩 묶어 1인 1행으로 ──
data_rows = raw.iloc[8:].reset_index(drop=True)
records = []

for i in range(0, len(data_rows), 2):
    row1 = data_rows.iloc[i]
    row2 = data_rows.iloc[i + 1] if i + 1 < len(data_rows) else pd.Series([None] * len(row1))

    record = {}
    for col_idx in range(len(col_row1)):
        name1 = col_row1[col_idx]
        name2 = col_row2[col_idx]
        val1 = row1.iloc[col_idx] if not pd.isna(row1.iloc[col_idx]) else None
        val2 = row2.iloc[col_idx] if not pd.isna(row2.iloc[col_idx]) else None

        if name1 and name2:
            record[name1] = val1
            record[name2] = val2
        elif name1:
            record[name1] = val1
        elif name2:
            record[name2] = val2

    records.append(record)

df_biz_emp = pd.DataFrame(records)

print(f"\n변환 완료: {df_biz_emp.shape[0]}행 × {df_biz_emp.shape[1]}열")
print("컬럼:", df_biz_emp.columns.tolist())
display(df_biz_emp)

제목: 사 업  소 득 자  명 부
기준일: 2026-03-12
회사명: 서울유캔수학학원

1행 헤더: ['NO', '코드', '성명', '소득구분', '상호', '사업장주소', '우편번호']
2행 헤더: ['', '내/외국인', '주민(외국인)등록번호', '거주구분', '사업자등록번호', '소득자주소', '']

변환 완료: 21행 × 12열
컬럼: ['NO', '코드', '내/외국인', '성명', '주민(외국인)등록번호', '소득구분', '거주구분', '상호', '사업자등록번호', '사업장주소', '소득자주소', '우편번호']


,NO,코드,내/외국인,성명,주민(외국인)등록번호,소득구분,거주구분,상호,사업자등록번호,사업장주소,소득자주소,우편번호
0,1,000001,내국인,서지은,041011-4057221,기타자영업,거주,None,None,None,None,None
1,2,000002,내국인,이재윤,010327-3174310,기타자영업,거주,None,None,None,None,None
2,3,000003,내국인,위민정,011211-4212411,기타자영업,거주,None,None,None,None,None
3,4,000004,내국인,남미현,810702-2249118,학원강사,거주,None,None,None,None,None
4,5,000005,내국인,백경관,820613-1622414,학원강사,거주,None,None,None,None,None
5,6,000006,내국인,김선례,760616-2080519,학원강사,거주,None,None,None,None,None
6,7,000007,내국인,전영수,830126-1080819,학원강사,거주,None,None,None,None,None
7,8,000008,내국인,남궁찬,031028-3066718,학원강사,거주,None,None,None,None,None
8,9,000009,내국인,노희필,830418-1076332,기타자영업,거주,None,None,None,None,None
9,10,000010,내국인,전미연,970224-2081221,기타자영업,거주,None,None,None,None,None


### 기타 소득 직원

In [12]:
from pathlib import Path
import pandas as pd

# ── 파일 경로 ──
etc_emp_file_name = "기타소득자명부.xls"
etc_emp_file_path = Path("wehago") / etc_emp_file_name

if not etc_emp_file_path.exists():
    candidates = list(Path('.').rglob(etc_emp_file_name))
    if candidates:
        etc_emp_file_path = candidates[0]
        print(f"파일을 찾았습니다: {etc_emp_file_path}")
    else:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {etc_emp_file_name}")

raw = pd.read_excel(etc_emp_file_path, engine="xlrd", header=None)

# ── 메타 정보 추출 ──
title = str(raw.iloc[0, 3]).strip()       # 기타소득자명부
report_date = str(raw.iloc[2, 3]).strip() # 2026년 03월 12일 현재
company = str(raw.iloc[4, 0]).strip()     # 종로구육아종합지원센터회사명:
print(f"제목: {title}")
print(f"기준일: {report_date}")
print(f"회사명: {company}\n")

# ── 2행 헤더 (Row 5-6) ──
h1 = raw.iloc[5].tolist()  # 코드, 성명, 주민(외국인)등록번호, 사업자등록번호, 사업장주소, 우편번호
h2 = raw.iloc[6].tolist()  # 내/외국인, 소득구분, 거주구분, 법인/개인구분, 전화번호, (빈)

def _clean(v):
    if pd.isna(v):
        return ""
    return str(v).strip().replace(" ", "")

col_row1 = [_clean(v) for v in h1]
col_row2 = [_clean(v) for v in h2]

print("1행 헤더:", col_row1)
print("2행 헤더:", col_row2)

# ── 데이터 영역 (Row 7~) 2행씩 묶어 1인 1행으로 ──
data_rows = raw.iloc[7:].reset_index(drop=True)
records = []

for i in range(0, len(data_rows), 2):
    row1 = data_rows.iloc[i]
    row2 = data_rows.iloc[i + 1] if i + 1 < len(data_rows) else pd.Series([None] * len(row1))

    record = {}
    for col_idx in range(len(col_row1)):
        name1 = col_row1[col_idx]
        name2 = col_row2[col_idx]
        val1 = row1.iloc[col_idx] if not pd.isna(row1.iloc[col_idx]) else None
        val2 = row2.iloc[col_idx] if not pd.isna(row2.iloc[col_idx]) else None

        if name1 and name2:
            record[name1] = val1
            record[name2] = val2
        elif name1:
            record[name1] = val1
        elif name2:
            record[name2] = val2

    records.append(record)

df_etc_emp = pd.DataFrame(records)

print(f"\n변환 완료: {df_etc_emp.shape[0]}행 × {df_etc_emp.shape[1]}열")
print("컬럼:", df_etc_emp.columns.tolist())
display(df_etc_emp)

제목: 기  타  소  득  자  명  부
기준일: 2026년 03월 12일 현재
회사명: 종로구육아종합지원센터회사명:

1행 헤더: ['코드', '성명', '주민(외국인)등록번호', '사업자등록번호', '사업장주소', '우편번호']
2행 헤더: ['내/외국인', '소득구분', '거주구분', '법인/개인구분', '전화번호', '']

변환 완료: 206행 × 11열
컬럼: ['코드', '내/외국인', '성명', '소득구분', '주민(외국인)등록번호', '거주구분', '사업자등록번호', '법인/개인구분', '사업장주소', '전화번호', '우편번호']


,코드,내/외국인,성명,소득구분,주민(외국인)등록번호,거주구분,사업자등록번호,법인/개인구분,사업장주소,전화번호,우편번호
0,000001,내국인,고익환,그 밖에 필요경비 있는 기타소득(64/68/69/71~77/79 제외),520405-1673612,거주,NaN,개인,"대구광역시 수성구 달구벌대로 2435 109-3106 (범어동, 두산위브더제니스)",,42028
1,000002,내국인,권오철,그 밖에 필요경비 있는 기타소득(64/68/69/71~77/79 제외),720130-1810119,거주,NaN,개인,"경기도 부천시 중동로248번길 52 네이버시티빌딩 506호 (중동, 네이버시티빌딩)",,14548
2,000003,내국인,김도형,그 밖에 필요경비 있는 기타소득(64/68/69/71~77/79 제외),800502-1029715,거주,NaN,개인,"인천광역시 남동구 논현로 235 1002-1003 (논현동, 별빛마을웰카운티아파트)",,21661
3,000004,내국인,김동례62,그 밖에 필요경비 있는 기타소득(64/68/69/71~77/79 제외),670928-2551016,거주,NaN,개인,"광주광역시 광산구 목련로21번길 20 105-1705 (산정동, 1060번지)",,62335
4,000005,내국인,김보미,그 밖에 필요경비 있는 기타소득(64/68/69/71~77/79 제외),880902-2066111,거주,NaN,개인,경기도 고양시 덕양구 행신동 769번지 무원마을 동신아파트 302동 602호,,412-220
...,...,...,...,...,...,...,...,...,...,...,...
201,000202,내국인,김상옥,강연료 등,730320-2624611,거주,NaN,개인,NaN,NaN,NaN
202,000203,내국인,김승옥,강연료 등,721019-2036821,거주,NaN,개인,NaN,NaN,NaN
203,000204,내국인,임재문,강연료 등,751103-1052416,거주,NaN,개인,NaN,NaN,NaN
204,000205,내국인,이은경,강연료 등,890911-2677810,거주,NaN,개인,NaN,NaN,NaN


### 일용직 소득 직원